In [1]:
from pathlib import Path
import random

import numpy as np
import yaml

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("project root:", PROJECT_ROOT)
print("raw data dir:", DATA_RAW)


project root: /Users/m/Documents/capstone-msc-da-feb-25-cohort-MeerimZhenisbekova
raw data dir: /Users/m/Documents/capstone-msc-da-feb-25-cohort-MeerimZhenisbekova/data/raw


In [2]:
WAID_DIR = DATA_RAW / "waid"
assert WAID_DIR.exists(), f"WAID folder not found at {WAID_DIR}"

waid_yaml = next(WAID_DIR.rglob("data.yaml"), None)
assert waid_yaml is not None, "data.yaml not found under the WAID folder"

with open(waid_yaml) as f:
    waid_meta = yaml.safe_load(f)

print("WAID data.yaml:", waid_yaml)
print("WAID class names:", waid_meta["names"])

waid_root = waid_yaml.parent
for split in ("train", "valid", "test"):
    imgs_dir = waid_root / split / "images"
    n = len(list(imgs_dir.glob("*.jpg"))) if imgs_dir.exists() else 0
    print(f"  {split}: {n} images")


WAID data.yaml: /Users/m/Documents/capstone-msc-da-feb-25-cohort-MeerimZhenisbekova/data/raw/waid/data.yaml
WAID class names: ['camelus', 'cattle', 'kiang', 'seal', 'sheep', 'zebra']
  train: 10056 images
  valid: 2873 images
  test: 1437 images


In [3]:
import fiftyone as fo
import fiftyone.zoo as foz

COCO_DIR = DATA_RAW / "coco_cow_sheep"
COCO_DIR.mkdir(parents=True, exist_ok=True)

coco_exported = COCO_DIR / "exported"

if not coco_exported.exists():
    ds = foz.load_zoo_dataset(
        "coco-2017",
        split="train",
        label_types=["detections"],
        classes=["cow", "sheep"],
        max_samples=3500,
        shuffle=True,
        seed=SEED,
        dataset_name="coco_cow_sheep_clean",
        drop_existing_dataset=True,
    )
    ds.export(
        export_dir=str(coco_exported),
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        classes=["cow", "sheep"],
    )

print("COCO exported to", coco_exported)
for p in sorted(coco_exported.glob("*")):
    print(" ", p.name)


Found annotations at '/Users/m/fiftyone/coco-2017/raw/instances_train2017.json'


Only found 3393 (<3500) samples matching your requirements


Sufficient images already downloaded


Existing download of split 'train' is sufficient


Loading 'coco-2017' split 'train'


...

  97% |██████████████|| 3295/3393 [4.1s elapsed, 114.2ms remaining, 859.7 samples/s] 

 100% |██████████████/| 3380/3393 [4.2s elapsed, 15.0ms remaining, 867.0 samples/s]  

 100% |███████████████| 3393/3393 [4.2s elapsed, 0s remaining, 871.3 samples/s]      


COCO exported to /Users/m/Documents/capstone-msc-da-feb-25-cohort-MeerimZhenisbekova/data/raw/coco_cow_sheep/exported
  dataset.yaml
  images
  labels


/opt/anaconda3/envs/stable/lib/python3.11/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'sandwich' not in provided classes
  warnings.warn(msg)


In [4]:
print("=== WAID ===")
print("classes:", waid_meta["names"])
for split in ("train", "valid", "test"):
    imgs = waid_root / split / "images"
    lbls = waid_root / split / "labels"
    n_img = len(list(imgs.glob("*.jpg"))) if imgs.exists() else 0
    n_lbl = len(list(lbls.glob("*.txt"))) if lbls.exists() else 0
    print(f"  {split}: images={n_img}, labels={n_lbl}")

print()
print("=== COCO cow + sheep subset ===")
coco_yaml = coco_exported / "dataset.yaml"
if coco_yaml.exists():
    with open(coco_yaml) as f:
        coco_meta = yaml.safe_load(f)
    print("classes:", coco_meta["names"])

for subdir in ("images", "labels"):
    for split in ("train", "val", "test"):
        d = coco_exported / subdir / split
        n = len(list(d.glob("*"))) if d.exists() else 0
        if n > 0:
            print(f"  {subdir}/{split}: {n} files")


=== WAID ===
classes: ['camelus', 'cattle', 'kiang', 'seal', 'sheep', 'zebra']
  train: images=10056, labels=10056


  valid: images=2873, labels=2873
  test: images=1437, labels=1437

=== COCO cow + sheep subset ===
classes: {0: 'cow', 1: 'sheep'}
  images/val: 3393 files
  labels/val: 3393 files
